In [3]:
# Instalación
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-rls07asb/unsloth_6e9e98165b394b2aafd4108535304e89
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-rls07asb/unsloth_6e9e98165b394b2aafd4108535304e89
  Resolved https://github.com/unslothai/unsloth.git to commit 64e4d52d4886740ab1b0ceef0e24e0707eeb9078
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [4]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
import getpass
from huggingface_hub import login, whoami
import pandas as pd

# 1. AUTENTICACIÓN Y CONFIGURACIÓN DEL MODELO

login(token=getpass.getpass("HF token (write): "))
HF_USER = whoami()["name"]
print("Conectado como:", HF_USER)

MODEL_SIZE = "mini"  # "mini" = 2B (T4 free)

MODELS = {
    "mini": "unsloth/Qwen3.5-2B",  # ~2B
    "small": "unsloth/Qwen3.5-9B",  # 9B
    "big": "unsloth/Qwen3.5-27B",  # 27B
}

model_name = MODELS[MODEL_SIZE]
print("Modelo elegido:", model_name)

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

@torch.no_grad()
def ask(question, max_new_tokens=120):
    model.eval()

    messages = [{"role": "user", "content": question}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text=text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return answer.strip()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
HF token (write): ··········
Conectado como: vari96
Modelo elegido: unsloth/Qwen3.5-2B
==((====))==  Unsloth 2026.9.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [5]:
# 2. CARGA, MAPEADO Y BALANCEO DEL DATASET LOCAL
# Carga de archivo CSV
df = pd.read_csv("sentiment_mental_health.csv")

# Eliminación de categorías (Personality disorder y Bipolar)
df = df[~df['status'].isin(['Personality disorder', 'Bipolar'])].copy()

# Agrupación de categorías
def map_status(status):
    if status in ['Anxiety', 'Stress']:
        return 'Anxiety'
    elif status in ['Depression', 'Suicidal']:
        return 'Depression'
    elif status == 'Normal':
        return 'Normal'
    return status

df['status'] = df['status'].apply(map_status)

# Convertir a Dataset de Hugging Face
dataset = Dataset.from_pandas(df)

# BALANCEO EQUITATIVO:
# Se obtiene el número mínimo de registros entre las 3 categorías
# y se selecciona esa cantidad por cada categoría.
min_count = min([len(dataset.filter(lambda x: x["status"] == cat)) for cat in ["Normal", "Depression", "Anxiety"]])
print(f"Número de muestras balanceadas por categoría: {min_count}")

categorias = ["Normal", "Depression", "Anxiety"]
dataset = concatenate_datasets([
    dataset.filter(lambda x: x["status"] == cat).select(range(min_count)) for cat in categorias
]).shuffle(seed=42)

print(f"Total de registros balanceados: {len(dataset)}")


Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Número de muestras balanceadas por categoría: 6428


Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48827 [00:00<?, ? examples/s]

Total de registros balanceados: 19284


In [6]:
# 3. FORMATEO PROMPT / CHAT TEMPLATE

def formatting_prompts_func(examples):
    textos = examples["statement"]
    estados = examples["status"]
    formatted_texts = []

    for texto, estado in zip(textos, estados):
        messages = [
            {
                "role": "user",
                "content": f"Classify the following text into one of these mental health categories (Normal, Depression, Anxiety):\n\nText: {texto}"
            },
            {
                "role": "assistant",
                "content": f"Category: {estado}"
            }
        ]

        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize = False,
            add_generation_prompt = False
        )
        formatted_texts.append(formatted)

    return { "text": formatted_texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset[0]["text"])


Map:   0%|          | 0/19284 [00:00<?, ? examples/s]

<|im_start|>user
Classify the following text into one of these mental health categories (Normal, Depression, Anxiety):

Text: i haven't struggle in almost a year but here i am here's the deal: i have a very real cyst. it is possibly to likely infected. it's just one of those benign ones you get on your skin. well i've had mine for four to five years. no big deal. this last week it has started to hurt and become infected. lame. it's rather large. like an inch diameter. and i cannot stop worrying i'm about to contract sepsis and die. like it is rather painful. i went to a UC and they said i was fine. i have a dermatology appointment in a week. but i'm still freaking out. i will be fine. i know i will. i just cannot chill out. i am trying so hard to take my mind off of it but it's on my lower butt cheek and basically any kind of sitting really aggravates it. idk what to do!! give me techniques to chill!! i've been really good about this until today<|im_end|>
<|im_start|>assistant
<think>


In [7]:
# 4. CONFIGURACIÓN Y ENTRENAMIENTO

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = getattr(tokenizer, "tokenizer", tokenizer),
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 512,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        dataloader_num_workers = 0,
        seed = 3407,
        report_to = "none",
        output_dir = "outputs",
    ),
)

trainer.train()


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/19284 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,284 | Num Epochs = 1 | Total steps = 2,411
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 16,819,200 of 2,230,060,864 (0.75% trained)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/pytho

Step,Training Loss
10,2.854338
20,2.348373
30,2.295812
40,2.395376
50,2.304016
60,2.322743
70,2.360104
80,2.402764
90,2.361820
100,2.379334


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2411/tokenizer_config.json.


TrainOutput(global_step=2411, training_loss=2.272181210701815, metrics={'train_runtime': 14345.5751, 'train_samples_per_second': 1.344, 'train_steps_per_second': 0.168, 'total_flos': 4.312580316839962e+16, 'train_loss': 2.272181210701815, 'epoch': 1.0})

In [8]:
# 5. PRUEBAS

# Prueba 1: Texto Ansiedad / Estrés
texto_prueba = "I feel like everything is overwhelming, my heart is beating fast and I feel extremely stressed."
print("Resultado:", ask(f"Classify the following text into one of these mental health categories (Normal, Depression, Anxiety):\n\nText: {texto_prueba}"))

# Prueba 2: Texto Normal
texto_prueba = "I had a productive day today and spent time with my friends."
print("Resultado:", ask(f"Classify the following text into one of these mental health categories (Normal, Depression, Anxiety):\n\nText: {texto_prueba}"))

# Prueba 3: Texto Depresión / Suicida
texto_prueba = "I feel hopeless, I can't get out of bed and I don't see any reason to continue."
print("Resultado:", ask(f"Classify the following text into one of these mental health categories (Normal, Depression, Anxiety):\n\nText: {texto_prueba}"))

Resultado: <think>

</think>

Category: Anxiety
Resultado: <think>

</think>

Category: Normal
Resultado: <think>

</think>

Category: Depression
